# NUMPY MASTERY — CHAPTERS 6–9

```
NUMPY MASTERY
Current Chapters: 6 (Boolean Indexing), 7 (Fancy Indexing),
                  8 (View vs Copy), 9 (Reshaping)
Progress:         9 / 75 chapters
Completed:        Ch1-5
Remaining:        66 chapters
```

These four chapters are grouped because they answer one connected question that causes more
production bugs than any other NumPy topic: **"When I select or reshape data, did I get an
independent copy, or a window into the original memory?"**


In [ ]:
import numpy as np
print("NumPy version:", np.__version__)


---
# CHAPTER 6 — Boolean Indexing

## Concept
A boolean array of the same shape as `arr` (or broadcastable to it) can be used as an index.
`arr[mask]` returns a **1D array of the elements where `mask` is `True`**.

## Why it matters
This is the primary tool for filtering rows, removing outliers, thresholding, and conditional
feature engineering — the numeric-array equivalent of a pandas `.loc[condition]`.

## Industry use case
Fraud detection: flag transactions where `amount > threshold`. Data cleaning: drop sensor
readings that are physically impossible (negative age, temperature below absolute zero).


In [ ]:
data = np.array([12, -5, 47, 0, 99, -1, 33])

mask = data > 10
print("mask        :", mask)
print("data[mask]  :", data[mask])                      # boolean indexing
print("filtered    :", data[(data > 0) & (data < 50)])  # combined condition
print("negated     :", data[~(data > 10)])               # NOT


### Expected Output
```
mask        : [ True False  True False  True False  True]
data[mask]  : [12 47 99 33]
filtered    : [12  0 33]
negated     : [-5  0 -1]
```

### `&`/`|`/`~` vs `and`/`or`/`not` — why you MUST use the bitwise operators
Python's `and`/`or`/`not` call `__bool__()` on their operands, which requires a **single**
boolean value. A NumPy array has *many* boolean values, so `arr > 10 and arr < 50` raises
`ValueError: truth value of an array with more than one element is ambiguous`.
`&`, `|`, `~` are element-wise bitwise operators that NumPy overloads to do element-wise
logical AND/OR/NOT — and they require careful parenthesization because `&` binds *tighter*
than comparison operators in Python, so `data > 0 & data < 50` (without parens) is a
precedence bug, not what you meant.

### `np.where()`, `np.nonzero()`, `np.argwhere()`
```python
np.where(data > 10)      # -> tuple of index arrays where condition is True
np.where(data > 10, data, 0)  # -> ternary: keep value if True, else 0 (same shape as data)
np.nonzero(data)         # -> indices of all non-zero elements
np.argwhere(data > 10)   # -> Nx1 array of indices (ND-friendly version of where)
```


In [ ]:
print("np.where(data > 10)              :", np.where(data > 10))
print("np.where(data > 10, data, 0)      :", np.where(data > 10, data, 0))
print("np.nonzero(data)                  :", np.nonzero(data))
print("np.argwhere(data > 10)            :\n", np.argwhere(data > 10))


### Expected Output
```
np.where(data > 10)              : (array([0, 2, 4, 6]),)
np.where(data > 10, data, 0)      : [12  0 47  0 99  0 33]
np.nonzero(data)                  : (array([0, 1, 2, 4, 5, 6]),)
np.argwhere(data > 10)            :
 [[0]
  [2]
  [4]
  [6]]
```

### Common Mistake
Writing `data[data > 0 & data < 50]` without parentheses — due to operator precedence this
actually evaluates as `data[data > (0 & data) < 50]`, which is nonsense and either errors or
silently gives the wrong result. **Always parenthesize each condition** when combining with
`&`/`|`.

### Interview Insight
> "Why can't you use `and`/`or` with NumPy boolean arrays?"
Because those Python keywords require coercing the whole array to a single `bool` via
`__bool__`, which is ambiguous for arrays with more than one element. `&`/`|`/`~` are
overloaded to operate element-wise instead.


---
# CHAPTER 7 — Fancy Indexing

## Concept
Indexing with an **array (or list) of integers**, rather than a slice, selects an arbitrary
(possibly repeated, possibly reordered) set of elements. This is called "fancy" or "advanced"
indexing.

## Why it matters
Reordering rows (e.g., shuffling a dataset), selecting a specific, non-contiguous subset of
features/rows (e.g., a mini-batch by index), or repeating rows/columns.


In [ ]:
arr = np.arange(10, 20)
print("arr:", arr)
print("arr[[1, 3, 5]]      :", arr[[1, 3, 5]])       # arbitrary order & selection
print("arr[[0, 0, 0]]      :", arr[[0, 0, 0]])       # repetition allowed
print("arr[[-1, -2]]       :", arr[[-1, -2]])        # negative indices work too

matrix = np.arange(1, 17).reshape(4, 4)
print("\nmatrix:\n", matrix)
print("matrix[[0, 2]]      :\n", matrix[[0, 2]])           # select rows 0 and 2
print("matrix[:, [0, 2]]   :\n", matrix[:, [0, 2]])        # select columns 0 and 2
print("matrix[[0,1],[0,1]] :", matrix[[0, 1], [0, 1]])      # (0,0) and (1,1) -- PAIRED indices!


### Expected Output
```
arr: [10 11 12 13 14 15 16 17 18 19]
arr[[1, 3, 5]]      : [11 13 15]
arr[[0, 0, 0]]      : [10 10 10]
arr[[-1, -2]]       : [19 18]

matrix:
 [[ 1  2  3  4]
  [ 5  6  7  8]
  [ 9 10 11 12]
  [13 14 15 16]]
matrix[[0, 2]]      :
 [[ 1  2  3  4]
  [ 9 10 11 12]]
matrix[:, [0, 2]]   :
 [[ 1  3]
  [ 5  7]
  [ 9 11]
  [13 15]]
matrix[[0,1],[0,1]] : [1 6]
```

### Critical gotcha: `matrix[[0,1],[0,1]]` does NOT select a 2x2 sub-block
This is the #1 fancy-indexing trap. `matrix[[0, 1], [0, 1]]` pairs the index arrays
element-wise: it picks `matrix[0,0]` and `matrix[1,1]` — **not** the rows-0&1-by-cols-0&1
sub-block. To get an actual sub-block, use `matrix[np.ix_([0,1],[0,1])]` or
`matrix[[0,1]][:, [0,1]]`.

### Fancy indexing ALWAYS returns a copy
Unlike basic slicing, fancy indexing (integer-array or boolean) **always** allocates a new
array — it cannot be expressed as a strided view because the selected elements aren't evenly
spaced in memory in general. This is the bridge to Chapter 8.

### Interview Insight
> "Does fancy indexing return a view or a copy? Why?"
Always a copy. A view requires the result to be expressible via a fixed stride pattern over the
original buffer; an arbitrary/possibly-repeated selection of indices generally can't be, so
NumPy must materialize new memory.


---
# CHAPTER 8 — View vs Copy

## Concept
This is the single most important "gotcha" chapter in practical NumPy work.

| Operation | View or Copy? |
|---|---|
| Basic slicing (`arr[1:5]`, `arr[:, 2]`, `arr[::2]`) | **View** (shares memory) |
| `.reshape()` (when possible) | **View** |
| `.T` / `.transpose()` | **View** |
| `.ravel()` | **View** (usually) |
| Fancy indexing (integer array) | **Copy** |
| Boolean indexing | **Copy** |
| `.flatten()` | **Copy** (always, by design) |
| Arithmetic (`arr + 1`) | **Copy** (new array) |
| `.copy()` | **Copy** (explicit, always) |

## Why it matters
A "view" means two Python variables point at the **same underlying memory buffer**. Mutating
one mutates the other — invisibly, at a distance, often across function boundaries. This is
either a powerful zero-copy optimization (when intentional) or a brutal bug (when accidental).


In [ ]:
original = np.arange(10)
print("original:", original)

view_slice = original[2:6]        # VIEW: shares memory
print("view_slice:", view_slice)
view_slice[0] = 999               # mutate the view...
print("original after mutating view_slice:", original, " <-- CHANGED!")

print()
original2 = np.arange(10)
copy_fancy = original2[[2, 3, 4, 5]]   # COPY: fancy indexing
copy_fancy[0] = 999
print("original2 after mutating copy_fancy:", original2, " <-- UNCHANGED")


### Expected Output
```
original: [0 1 2 3 4 5 6 7 8 9]
view_slice: [2 3 4 5]
original after mutating view_slice: [  0   1 999   3   4   5   6   7   8   9]  <-- CHANGED!

original2 after mutating copy_fancy: [0 1 2 3 4 5 6 7 8 9]  <-- UNCHANGED
```

### How to check programmatically: `.base`, `.flags.owndata`, `np.shares_memory()`


In [ ]:
original = np.arange(10)
view_slice = original[2:6]
copy_explicit = original[2:6].copy()

print("view_slice.base is original           :", view_slice.base is original)
print("view_slice.flags.owndata               :", view_slice.flags['OWNDATA'])
print("copy_explicit.flags.owndata            :", copy_explicit.flags['OWNDATA'])
print("np.shares_memory(original, view_slice) :", np.shares_memory(original, view_slice))
print("np.shares_memory(original, copy_explicit):", np.shares_memory(original, copy_explicit))


### Expected Output
```
view_slice.base is original           : True
view_slice.flags.owndata               : False
copy_explicit.flags.owndata            : True
np.shares_memory(original, view_slice) : True
np.shares_memory(original, copy_explicit): False
```

- `.base` — for a view, points back to the original array it was derived from (`None` if the
  array owns its data).
- `.flags['OWNDATA']` — `False` means this array is a view onto someone else's buffer.
- `np.shares_memory(a, b)` — the definitive, general-purpose check (exact answer, can be slow);
  `np.may_share_memory(a, b)` is a faster, conservative (may false-positive) check.

### Why this matters in production ML pipelines
```python
def normalize(X):
    X -= X.mean(axis=0)   # BUG: mutates the CALLER's array in-place if X is a view/slice!
    X /= X.std(axis=0)
    return X
```
If `X` passed in is a slice of a larger dataset (a view), this function silently corrupts data
the caller still holds a reference to elsewhere in the pipeline — a bug that can surface far
away from its cause, sometimes only under specific data-loading code paths.

### Best Practice
- When writing a function that must not mutate the caller's data, start with
  `X = np.array(X, copy=True)` (or `X = X.copy()`).
- When you *want* zero-copy performance and control mutation carefully, document that the
  function mutates in place, and prefer explicit `out=` parameters where ufuncs support them.

### Debugging Challenge
```python
def get_top_features(X, indices):
    return X[:, indices]

X = np.random.default_rng(0).normal(size=(5, 8))
top = get_top_features(X, [0, 1, 2])
top[:] = 0
# Q: Did this corrupt X? Why or why not? (Answer depends on whether `indices` is a list/array
#    -- fancy indexing -- vs a slice.)
```

### Interview Insight
> "Does `reshape()` return a view or a copy?"
It returns a view **when possible** (i.e., when the new shape is compatible with the existing
strides/memory layout without rearranging data) and silently falls back to a copy otherwise
(e.g., reshaping a non-contiguous array). This is exactly why Chapter 9 exists — you should
never *assume* which one you get without checking `.base` or `np.shares_memory`.


---
# CHAPTER 9 — Reshaping

## Concept
`reshape()`, `resize()`, `ravel()`, `flatten()`, `squeeze()`, `expand_dims()` all change an
array's shape without changing its data (the *values* stay the same; only how they're
organized into axes changes).

## Why it matters
Feeding data into models almost always requires reshaping: flattening an image into a feature
vector, adding a batch dimension, or turning a 1D prediction array into a column vector.


In [ ]:
arr = np.arange(12)
print("original       :", arr, arr.shape)

reshaped = arr.reshape(3, 4)
print("reshape(3, 4)  :\n", reshaped)

# reshaped_auto = arr.reshape(2, -1)   # -1 means "infer this dimension"
print("reshape(2, -1) :\n", reshaped_auto, " shape:", reshaped_auto.shape)

flat_back = reshaped.ravel()         # view where possible
print("ravel()        :", flat_back, "  shares memory with reshaped:", np.shares_memory(flat_back, reshaped))

flat_copy = reshaped.flatten()       # always a copy
print("flatten()      :", flat_copy, "  shares memory with reshaped:", np.shares_memory(flat_copy, reshaped))


### Expected Output
```
original       : [ 0  1  2  3  4  5  6  7  8  9 10 11] (12,)
reshape(3, 4)  :
 [[ 0  1  2  3]
  [ 4  5  6  7]
  [ 8  9 10 11]]
reshape(2, -1) :
 [[ 0  1  2  3  4  5]
  [ 6  7  8  9 10 11]]  shape: (2, 6)
ravel()        : [ 0  1  2  3  4  5  6  7  8  9 10 11]   shares memory with reshaped: True
flatten()      : [ 0  1  2  3  4  5  6  7  8  9 10 11]   shares memory with reshaped: False
```

### `reshape` vs `resize`
- `arr.reshape(new_shape)` — **requires** `new_shape` to have the same total number of
  elements; returns a view when possible, otherwise a copy. Does not modify `arr` in place
  (unless you reassign).
- `arr.resize(new_shape)` — modifies the array **in place**, and can change the *total* number
  of elements (padding with zeros if growing, truncating if shrinking) — genuinely different
  semantics, not just an in-place `reshape`.

### `ravel` vs `flatten`
Both collapse an array to 1D. `ravel()` returns a view when the array's memory layout allows it
(fast, no copy); `flatten()` **always** returns a copy (safer default, but always pays the copy
cost). Use `ravel()` when you don't intend to mutate the result and want to avoid an
unnecessary copy; use `flatten()` when you specifically need an independent array.

### `squeeze()` and `expand_dims()`
```python
x = np.array([[[1, 2, 3]]])     # shape (1, 1, 3)
x.squeeze()                     # shape (3,) -- removes ALL size-1 axes
x.squeeze(axis=0)               # shape (1, 3) -- removes only axis 0

y = np.array([1, 2, 3])         # shape (3,)
np.expand_dims(y, axis=0)       # shape (1, 3) -- e.g., add a batch dimension
np.expand_dims(y, axis=1)       # shape (3, 1) -- e.g., turn into a column vector
```

### Real-world use
- `expand_dims(image, axis=0)` — turn a single `(H, W, C)` image into a batch of 1:
  `(1, H, W, C)`, which is what most model `.predict()` APIs expect.
- `reshape(-1, 1)` — turn a 1D array of predictions into a column vector, often required by
  scikit-learn APIs that expect 2D input.
- `flatten()` — turn a `(H, W)` image patch into a 1D feature vector for a classical ML model.

### Common Mistake
Calling `.reshape(-1)` and assuming you always get a *view* — you don't, if the array isn't
contiguous (e.g., after a non-trivial transpose + slice combination). Check
`np.shares_memory()` if it matters for your use case (e.g., you're about to mutate in place for
performance).

### Interview Insight
> "What does `-1` mean in `reshape`?"
"Infer this dimension automatically from the total element count and the other specified
dimensions." Only one `-1` is allowed per call — NumPy can't infer two unknowns simultaneously.


---
## Chapters 6–9 Summary

- **Boolean indexing**: `arr[mask]` filters by condition; must use `&`/`|`/`~` with parens, not
  `and`/`or`/`not`. `np.where`, `np.nonzero`, `np.argwhere` give index-level control.
- **Fancy indexing**: integer-array indexing selects/reorders/repeats arbitrary elements;
  `matrix[[0,1],[0,1]]` pairs indices element-wise (NOT a sub-block — use `np.ix_` for that).
- **View vs Copy**: basic slicing/reshape/transpose/ravel → views (share memory) when possible;
  fancy/boolean indexing/flatten/arithmetic/`.copy()` → always copies. Check with `.base`,
  `.flags['OWNDATA']`, `np.shares_memory()` — never assume.
- **Reshaping**: `reshape` (view-if-possible, same element count) vs `resize` (in-place, can
  change count); `ravel` (view-if-possible) vs `flatten` (always copy); `squeeze`/`expand_dims`
  manage size-1 axes — essential for batch dimensions and scikit-learn's 2D-input APIs.

---

## Mastery Exercises

**Exercise 1 (Prediction):** For `matrix = np.arange(16).reshape(4,4)`, predict whether each of
these returns a view or a copy, then verify with `np.shares_memory`:
`matrix[1:3, 1:3]`, `matrix[[1,2], :]`, `matrix[matrix > 5]`, `matrix.T`, `matrix.reshape(2,8)`.

**Exercise 2:** Given a `(100, 20)` feature matrix `X`, write the indexing expression to select
only the rows where feature column 0 is greater than its own mean.

**Coding Challenge:** Implement `train_test_split_indices(n, test_ratio, seed)` that returns two
NumPy integer arrays of shuffled row indices (train, test) — no data copying of `X` itself, just
index arrays — using `np.random.default_rng(seed).permutation(n)`. This is the real technique
professional code uses instead of shuffling and copying huge feature matrices directly.

**Debugging Challenge:** A teammate's preprocessing function silently corrupts the training set
after being called on a validation subset. Given what you now know about views, write one
sentence identifying the most likely root cause pattern, without seeing their code.

**Interview Questions:**
1. Give three examples of NumPy operations that return views, and three that always copy.
2. Why does fancy indexing always copy while basic slicing usually doesn't?
3. What's the difference between `ravel()` and `flatten()`, and when would you deliberately
   choose the copying one?
4. `matrix[[0,1],[0,1]]` — what does this actually select, and how would you get the true 2x2
   sub-block instead?

**Quiz:**
- Q1: True/False — `arr[arr > 0]` shares memory with `arr`.
- Q2: What does `.flags['OWNDATA']` tell you?
- Q3: Which function lets you insert a new axis of size 1 at a specific position?

---

